<a href="https://colab.research.google.com/github/77MJohn77/Geospatial-Research-Portfolio/blob/main/drought_analysis_BRW_2019_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 Drought Analysis: Bilate River Watershed (BRW) — 2019 to 2025
**Using VCI, TCI, VHI via Google Earth Engine**

---
**Study Area:** `projects/dotted-music-307621/assets/Niger`  
**GEE Project:** `dotted-music-307621`  
**Indices:** Vegetation Condition Index (VCI), Temperature Condition Index (TCI), Vegetation Health Index (VHI)  
**Period:** May–August, 2019–2025  
**Data Sources:** MOD13Q1 (NDVI, 250m), MOD11A2 (LST, 1km), CHIRPS (Rainfall), SRTM (DEM)

---
### Drought Classification (Kogan 1995 / Bhuiyan et al. 2017)
| Class | VCI / TCI / VHI |
|---|---|
| Extreme Drought | < 10 |
| Severe Drought | 10–20 |
| Moderate Drought | 20–30 |
| Mild Drought | 30–40 |
| No Drought | > 40 |

## ⚙️ Step 1: Install & Authenticate Google Earth Engine

In [ ]:
# Install required packages
!pip install earthengine-api geemap matplotlib numpy pandas scipy pysal libpysal esda geopandas folium branca --quiet

import ee
import geemap
import geemap.colormaps as cm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import json, os, io, warnings
from scipy import stats
from IPython.display import display, HTML
import ipywidgets as widgets

warnings.filterwarnings('ignore')

# Authenticate and initialize GEE
try:
    ee.Initialize(project='dotted-music-307621')
    print('✅ GEE initialized successfully!')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='dotted-music-307621')
    print('✅ GEE authenticated and initialized!')

## 🗺️ Step 2: Load Study Area & Configuration

In [ ]:
# ─── CONFIGURATION ───────────────────────────────────────────────────────────
PROJECT_ID  = 'dotted-music-307621'
ASSET_PATH  = 'projects/dotted-music-307621/assets/Niger'
YEARS       = list(range(2019, 2026))          # 2019–2025
MONTHS      = {'May': 5, 'June': 6, 'July': 7, 'August': 8}
VHI_ALPHA   = 0.5                              # VHI weighting constant
OUTPUT_DIR  = 'drought_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drought class thresholds & colors
DROUGHT_CLASSES = {
    'Extreme':  {'range': (0,  10),  'color': '#d73027', 'code': 1},
    'Severe':   {'range': (10, 20),  'color': '#fc8d59', 'code': 2},
    'Moderate': {'range': (20, 30),  'color': '#fee090', 'code': 3},
    'Mild':     {'range': (30, 40),  'color': '#e0f3f8', 'code': 4},
    'No Drought':{'range': (40,100), 'color': '#1a9850', 'code': 5},
}

# ─── LOAD STUDY AREA ─────────────────────────────────────────────────────────
aoi = ee.FeatureCollection(ASSET_PATH)
aoi_geom = aoi.geometry()

# Quick preview
Map = geemap.Map(center=[7.5, 38.0], zoom=8)
Map.addLayer(aoi, {'color': 'blue'}, 'Study Area (BRW)')
Map.centerObject(aoi)
Map


## 📡 Step 3: Define Index Computation Functions

In [ ]:
# ─── TIME SERIES BASELINES (2000–2022 full period) ───────────────────────────
def get_baselines(month_num):
    """Compute per-pixel min/max NDVI and LST over the baseline period 2000–2022."""
    start = f'2000-{month_num:02d}-01'
    end   = f'2022-{month_num:02d}-28'

    ndvi_col = (ee.ImageCollection('MODIS/006/MOD13Q1')
                .filterDate(start, end)
                .filter(ee.Filter.calendarRange(month_num, month_num, 'month'))
                .filterBounds(aoi_geom)
                .select('NDVI')
                .map(lambda img: img.multiply(0.0001).clip(aoi_geom)))

    lst_col  = (ee.ImageCollection('MODIS/006/MOD11A2')
                .filterDate(start, end)
                .filter(ee.Filter.calendarRange(month_num, month_num, 'month'))
                .filterBounds(aoi_geom)
                .select('LST_Day_1km')
                .map(lambda img: img.multiply(0.02).subtract(273.15).clip(aoi_geom)))

    return {
        'ndvi_min': ndvi_col.min(),
        'ndvi_max': ndvi_col.max(),
        'lst_min':  lst_col.min(),
        'lst_max':  lst_col.max(),
    }


def compute_indices(year, month_name, month_num, baselines):
    """Compute VCI, TCI, VHI for a given year/month."""
    date_start = f'{year}-{month_num:02d}-01'
    date_end   = f'{year}-{month_num:02d}-28'

    ndvi = (ee.ImageCollection('MODIS/006/MOD13Q1')
            .filterDate(date_start, date_end)
            .filterBounds(aoi_geom)
            .select('NDVI')
            .mean()
            .multiply(0.0001)
            .clip(aoi_geom))

    lst  = (ee.ImageCollection('MODIS/006/MOD11A2')
            .filterDate(date_start, date_end)
            .filterBounds(aoi_geom)
            .select('LST_Day_1km')
            .mean()
            .multiply(0.02).subtract(273.15)
            .clip(aoi_geom))

    b = baselines
    vci = (ndvi.subtract(b['ndvi_min'])
               .divide(b['ndvi_max'].subtract(b['ndvi_min']))
               .multiply(100)
               .rename('VCI'))

    tci = (b['lst_max'].subtract(lst)
               .divide(b['lst_max'].subtract(b['lst_min']))
               .multiply(100)
               .rename('TCI'))

    vhi = (vci.multiply(VHI_ALPHA)
               .add(tci.multiply(1 - VHI_ALPHA))
               .rename('VHI'))

    return {'VCI': vci, 'TCI': tci, 'VHI': vhi, 'NDVI': ndvi, 'LST': lst}


def classify_drought(index_img):
    """Classify an index image into 5 drought classes."""
    cls = (index_img
           .where(index_img.lt(10),  1)
           .where(index_img.gte(10).And(index_img.lt(20)), 2)
           .where(index_img.gte(20).And(index_img.lt(30)), 3)
           .where(index_img.gte(30).And(index_img.lt(40)), 4)
           .where(index_img.gte(40), 5))
    return cls.rename('DroughtClass')


print('✅ Index computation functions defined.')


## 🔄 Step 4: Compute All Indices (2019–2025)

In [ ]:
# ─── Pre-compute baselines per month (cached) ────────────────────────────────
print('⏳ Computing baselines per month (this may take ~2 min)...')
BASELINES = {}
for mname, mnum in MONTHS.items():
    BASELINES[mname] = get_baselines(mnum)
    print(f'  ✅ Baseline ready: {mname}')

# ─── Compute indices for every year × month ───────────────────────────────────
print('\n⏳ Computing VCI / TCI / VHI for 2019–2025...')
ALL_INDICES = {}  # ALL_INDICES[year][month_name] = {VCI, TCI, VHI, NDVI, LST}

for year in YEARS:
    ALL_INDICES[year] = {}
    for mname, mnum in MONTHS.items():
        ALL_INDICES[year][mname] = compute_indices(year, mname, mnum, BASELINES[mname])
    print(f'  ✅ {year} done')

print('\n✅ All indices computed!')


## 🗺️ Step 5: Professional Maps per Year & Month (Downloadable)

In [ ]:
# ─── VIS PARAMETERS ──────────────────────────────────────────────────────────
VIS = {
    'VCI': {'min': 0, 'max': 100,
            'palette': ['#d73027','#fc8d59','#fee090','#e0f3f8','#1a9850']},
    'TCI': {'min': 0, 'max': 100,
            'palette': ['#d73027','#fc8d59','#fee090','#e0f3f8','#1a9850']},
    'VHI': {'min': 0, 'max': 100,
            'palette': ['#d73027','#fc8d59','#fee090','#e0f3f8','#1a9850']},
}

CLASS_VIS = {
    'min': 1, 'max': 5,
    'palette': ['#d73027','#fc8d59','#fee090','#e0f3f8','#1a9850']
}

INDEX_LABELS = {'VCI': 'Vegetation Condition Index',
                'TCI': 'Temperature Condition Index',
                'VHI': 'Vegetation Health Index'}


def export_map_to_drive(img, description, folder='BRW_DroughtMaps'):
    """Export a GEE image to Google Drive."""
    task = ee.batch.Export.image.toDrive(
        image=img.clip(aoi_geom),
        description=description[:100],
        folder=folder,
        fileNamePrefix=description[:100],
        region=aoi_geom,
        scale=250,
        crs='EPSG:4326',
        maxPixels=1e10
    )
    task.start()
    return task


def make_professional_map(year, month_name, index_name, img, classified_img):
    """Create a professional static map with north arrow, legend, scale."""
    try:
        # Get thumbnail from GEE
        region = aoi_geom.bounds()
        thumb_url = img.visualize(**VIS[index_name]).getThumbURL({
            'region': region,
            'dimensions': 512,
            'format': 'png'
        })

        import requests
        from PIL import Image as PILImage
        resp = requests.get(thumb_url)
        img_arr = np.array(PILImage.open(io.BytesIO(resp.content)))

        fig, ax = plt.subplots(figsize=(8, 10))
        ax.imshow(img_arr)
        ax.axis('off')

        # ── Title
        ax.set_title(
            f'{INDEX_LABELS[index_name]} ({index_name})\n'
            f'{month_name} {year} — Bilate River Watershed, Ethiopia',
            fontsize=12, fontweight='bold', pad=10
        )

        # ── North Arrow
        ax.annotate('N', xy=(0.95, 0.92), xycoords='axes fraction',
                    ha='center', va='bottom', fontsize=14, fontweight='bold')
        ax.annotate('', xy=(0.95, 0.92), xytext=(0.95, 0.85),
                    xycoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

        # ── Legend
        legend_elements = [
            mpatches.Patch(facecolor='#d73027', label='Extreme (< 10)'),
            mpatches.Patch(facecolor='#fc8d59', label='Severe (10–20)'),
            mpatches.Patch(facecolor='#fee090', label='Moderate (20–30)'),
            mpatches.Patch(facecolor='#e0f3f8', label='Mild (30–40)'),
            mpatches.Patch(facecolor='#1a9850', label='No Drought (> 40)'),
        ]
        legend = ax.legend(
            handles=legend_elements,
            loc='lower left',
            title=f'{index_name} Drought Class',
            title_fontsize=9,
            fontsize=8,
            framealpha=0.9,
            edgecolor='black'
        )

        # ── Scale bar (approximate, ~50 km)
        ax.annotate('', xy=(0.25, 0.04), xytext=(0.05, 0.04),
                    xycoords='axes fraction',
                    arrowprops=dict(arrowstyle='|-|', color='black', lw=1.5))
        ax.text(0.15, 0.06, '~50 km', ha='center', va='bottom',
                transform=ax.transAxes, fontsize=8)

        # ── Data source
        fig.text(0.5, 0.01,
                 'Data: MODIS MOD13Q1 & MOD11A2 | Source: NASA LP DAAC via Google Earth Engine',
                 ha='center', fontsize=7, color='gray')

        plt.tight_layout()
        fname = f'{OUTPUT_DIR}/{index_name}_{year}_{month_name}.png'
        plt.savefig(fname, dpi=200, bbox_inches='tight')
        plt.show()
        print(f'  💾 Saved: {fname}')
        plt.close()

    except Exception as e:
        print(f'  ⚠️  Map render skipped ({e}) — check GEE connectivity')


print('✅ Mapping functions ready.')


In [ ]:
# ─── GENERATE & DISPLAY MAPS for every Year × Month × Index ──────────────────
# Also start GEE Drive exports

export_tasks = []

for year in YEARS:
    print(f'\n{'='*60}')
    print(f'  📅 YEAR: {year}')
    print(f'{'='*60}')
    for mname in MONTHS:
        for idx_name in ['VCI', 'TCI', 'VHI']:
            img = ALL_INDICES[year][mname][idx_name]
            cls_img = classify_drought(img)

            # Static professional map
            print(f'  🗺️  {idx_name} — {mname} {year}')
            make_professional_map(year, mname, idx_name, img, cls_img)

            # Export raw index image to Google Drive
            task = export_map_to_drive(
                img.rename(idx_name),
                description=f'{idx_name}_{year}_{mname}_BRW'
            )
            export_tasks.append((f'{idx_name}_{year}_{mname}', task))

            # Export classified drought map
            task2 = export_map_to_drive(
                cls_img.rename(f'{idx_name}_Class'),
                description=f'{idx_name}_Class_{year}_{mname}_BRW'
            )
            export_tasks.append((f'{idx_name}_Class_{year}_{mname}', task2))

print(f'\n✅ All maps generated. {len(export_tasks)} GEE export tasks submitted.')
print('📂 Check Google Drive → BRW_DroughtMaps folder for GeoTIFF downloads.')


## 📊 Step 6: Drought Area Coverage (%) — Per Year & Month

In [ ]:
# ─── Compute area fractions per drought class ─────────────────────────────────
def compute_area_percent(classified_img):
    """Return dict of drought class → area percent."""
    total_area = classified_img.gte(1).multiply(ee.Image.pixelArea())
    total = total_area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi_geom, scale=500, maxPixels=1e10
    ).getInfo().get('DroughtClass', 1)

    result = {}
    for cls_name, info in DROUGHT_CLASSES.items():
        mask = classified_img.eq(info['code'])
        area = (mask.multiply(ee.Image.pixelArea())
                .reduceRegion(reducer=ee.Reducer.sum(),
                              geometry=aoi_geom, scale=500,
                              maxPixels=1e10)
                .getInfo().get('DroughtClass', 0))
        result[cls_name] = round((area / total) * 100, 2) if total > 0 else 0
    return result


print('⏳ Computing area coverage statistics (this may take a few minutes)...')
AREA_STATS = {}  # [year][month][index] = {class: pct}

for year in YEARS:
    AREA_STATS[year] = {}
    for mname in MONTHS:
        AREA_STATS[year][mname] = {}
        for idx_name in ['VCI', 'TCI', 'VHI']:
            img = ALL_INDICES[year][mname][idx_name]
            cls_img = classify_drought(img)
            AREA_STATS[year][mname][idx_name] = compute_area_percent(cls_img)
    print(f'  ✅ {year}')

print('\n✅ Area statistics computed.')


In [ ]:
# ─── PLOT: Drought Area Coverage — VHI per month, all years ───────────────────
cls_colors = ['#d73027','#fc8d59','#fee090','#e0f3f8']
cls_names  = ['Extreme','Severe','Moderate','Mild']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('VHI Drought Area Coverage (%) — 2019–2025\nBilate River Watershed, Ethiopia',
             fontsize=14, fontweight='bold')

for ax, mname in zip(axes.flatten(), MONTHS):
    bottom = np.zeros(len(YEARS))
    for cls, col in zip(cls_names, cls_colors):
        vals = [AREA_STATS[y][mname]['VHI'].get(cls, 0) for y in YEARS]
        ax.bar(YEARS, vals, bottom=bottom, color=col, label=cls, width=0.6, edgecolor='white')
        bottom += np.array(vals)
    ax.set_title(f'{mname}', fontweight='bold')
    ax.set_ylabel('Area (%)')
    ax.set_xticks(YEARS)
    ax.set_xticklabels(YEARS, rotation=45)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.4)
    ax.legend(fontsize=7, loc='upper right')

plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/VHI_area_coverage_by_month.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: VHI_area_coverage_by_month.png')


## 📈 Step 7: Drought Class Trends — Extreme, Severe, Moderate

In [ ]:
# ─── Trend lines for VHI — Extreme / Severe / Moderate per month ──────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('VHI Drought Class Trends (2019–2025) — Bilate River Watershed',
             fontsize=13, fontweight='bold')

trend_classes = ['Extreme','Severe','Moderate']
trend_colors  = ['#d73027','#fc8d59','#fee090']
month_markers = {'May':'o','June':'s','July':'^','August':'D'}

for ax, cls, col in zip(axes, trend_classes, trend_colors):
    for mname, marker in month_markers.items():
        vals = [AREA_STATS[y][mname]['VHI'].get(cls, 0) for y in YEARS]
        ax.plot(YEARS, vals, marker=marker, label=mname,
                linewidth=1.8, markersize=6)
    ax.set_title(f'{cls} Drought', fontweight='bold', color=col)
    ax.set_xlabel('Year')
    ax.set_ylabel('Area (%)')
    ax.set_xticks(YEARS)
    ax.set_xticklabels(YEARS, rotation=45)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.4)

plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/VHI_drought_class_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: VHI_drought_class_trends.png')


In [ ]:
# ─── Annual Summary: Most Intense Drought Years (stacked bars per year) ───────
fig, ax = plt.subplots(figsize=(12, 6))

# Average over months for each year
bottom = np.zeros(len(YEARS))
for cls, col in zip(cls_names, cls_colors):
    avgs = []
    for y in YEARS:
        mean_val = np.mean([AREA_STATS[y][m]['VHI'].get(cls, 0) for m in MONTHS])
        avgs.append(mean_val)
    ax.bar(YEARS, avgs, bottom=bottom, color=col, label=cls, edgecolor='white', width=0.7)
    bottom += np.array(avgs)

ax.set_title('VHI Drought Class Area Coverage — Annual Average (May–Aug)\nBilate River Watershed, Ethiopia',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Average Area Coverage (%)')
ax.set_xticks(YEARS)
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/VHI_annual_drought_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: VHI_annual_drought_summary.png')


## 🔬 Step 8: Spatial Autocorrelation — Moran's I (VHI per Year)

In [ ]:
from libpysal.weights import lat2W
from esda.moran import Moran


def sample_vhi_values(img, n_points=200):
    """Sample VHI pixel values using stratified random points."""
    samples = img.sample(
        region=aoi_geom,
        scale=1000,
        numPixels=n_points,
        seed=42,
        geometries=True
    )
    data = samples.getInfo()['features']
    values = [f['properties'].get('VHI', np.nan) for f in data]
    coords = [(f['geometry']['coordinates'][0],
               f['geometry']['coordinates'][1]) for f in data]
    return values, coords


def compute_morans_i(values):
    """Compute Moran's I given a 1D array of values."""
    arr = np.array(values)
    arr = arr[~np.isnan(arr)]
    if len(arr) < 10:
        return None, None, None
    # Use a simple queen contiguity proxy via grid
    n = len(arr)
    side = int(np.sqrt(n))
    arr_sq = arr[:side*side].reshape(side, side)
    w = lat2W(side, side, rook=False)
    w.transform = 'r'
    mi = Moran(arr_sq.flatten(), w)
    return round(mi.I, 4), round(mi.z_norm, 4), round(mi.p_norm, 5)


print('⏳ Computing Moran\'s I for VHI (2019–2025)...')
MORANS = {}  # [year][month] = (I, Z, p)

for year in YEARS:
    MORANS[year] = {}
    for mname in MONTHS:
        vhi_img = ALL_INDICES[year][mname]['VHI']
        vals, _ = sample_vhi_values(vhi_img)
        I, Z, p = compute_morans_i(vals)
        MORANS[year][mname] = {'I': I, 'Z': Z, 'p': p}
    print(f'  ✅ {year}')

# Print summary table
print('\n📋 Moran\'s I Summary Table:')
rows = []
for year in YEARS:
    for mname in MONTHS:
        m = MORANS[year][mname]
        rows.append({'Year': year, 'Month': mname,
                     'Moran_I': m['I'], 'Z_score': m['Z'], 'p_value': m['p']})
morans_df = pd.DataFrame(rows)
morans_df.to_csv(f'{OUTPUT_DIR}/morans_i_summary.csv', index=False)
print(morans_df.to_string(index=False))


In [ ]:
# ─── PLOT Moran's I heatmap ────────────────────────────────────────────────────
pivot_I = morans_df.pivot(index='Year', columns='Month', values='Moran_I')
pivot_I = pivot_I[list(MONTHS.keys())]   # enforce month order

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot_I.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(MONTHS)))
ax.set_xticklabels(list(MONTHS.keys()), fontsize=11)
ax.set_yticks(range(len(YEARS)))
ax.set_yticklabels(YEARS, fontsize=11)

# Annotate cells
for i, year in enumerate(YEARS):
    for j, mname in enumerate(MONTHS):
        val = pivot_I.loc[year, mname]
        if val is not None and not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=10, fontweight='bold',
                    color='black' if val > 0.4 else 'white')

plt.colorbar(im, ax=ax, label="Moran's I")
ax.set_title("Spatial Autocorrelation — Moran's I for VHI\nBilate River Watershed, 2019–2025",
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Year')
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/morans_I_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: morans_I_heatmap.png')


In [ ]:
# ─── PLOT Moran's I scatterplot for worst drought year in dataset ──────────────
# Find year with highest average Moran's I
avg_I = {y: np.nanmean([MORANS[y][m]['I'] for m in MONTHS if MORANS[y][m]['I'] is not None])
         for y in YEARS}
worst_year = max(avg_I, key=avg_I.get)
print(f'Highest average Moran\'s I year: {worst_year} (avg I = {avg_I[worst_year]:.3f})')

# Plot Z-score bars per month for that year
z_vals = [MORANS[worst_year][m]['Z'] for m in MONTHS]
colors = ['#d73027' if z > 2.58 else '#fc8d59' if z > 1.96 else '#1a9850' for z in z_vals]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(list(MONTHS.keys()), z_vals, color=colors, edgecolor='white', width=0.5)
ax.axhline(2.58, color='red',    linestyle='--', linewidth=1.2, label='99% CI (z=2.58)')
ax.axhline(1.96, color='orange', linestyle='--', linewidth=1.2, label='95% CI (z=1.96)')
ax.set_title(f"Moran's I Z-scores — VHI {worst_year}\nBilate River Watershed",
             fontweight='bold')
ax.set_ylabel('Z-score')
ax.legend()
ax.grid(axis='y', alpha=0.4)

# Annotate Moran's I values
for bar, m in zip(bars, MONTHS):
    I_val = MORANS[worst_year][m]['I']
    if I_val:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.2,
                f'I={I_val:.2f}', ha='center', fontsize=9)

plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/morans_I_zscore_{worst_year}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Saved: morans_I_zscore_{worst_year}.png')


## 🔗 Step 9: Correlation Matrix — Drought Indices vs Environmental Variables

In [ ]:
# ─── Sample all variables at once for correlation ─────────────────────────────
def sample_all_vars(year, month_name, month_num, n=300):
    """Sample VCI, TCI, VHI, NDVI, LST, Rainfall, Temperature, Elevation."""
    date_s = f'{year}-{month_num:02d}-01'
    date_e = f'{year}-{month_num:02d}-28'

    indices  = ALL_INDICES[year][month_name]

    # CHIRPS rainfall
    rain = (ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD')
            .filterDate(date_s, date_e)
            .filterBounds(aoi_geom)
            .sum().rename('Rainfall').clip(aoi_geom))

    # DEM
    dem = ee.Image('USGS/SRTMGL1_003').select('elevation').rename('Altitude').clip(aoi_geom)

    # Stack all bands
    stack = (indices['VCI'].rename('VCI')
             .addBands(indices['TCI'].rename('TCI'))
             .addBands(indices['VHI'].rename('VHI'))
             .addBands(indices['NDVI'].rename('NDVI'))
             .addBands(indices['LST'].rename('LST'))
             .addBands(rain)
             .addBands(dem))

    pts = stack.sample(region=aoi_geom, scale=1000, numPixels=n, seed=42)
    rows = pts.getInfo()['features']
    records = [f['properties'] for f in rows]
    return pd.DataFrame(records)


print('⏳ Sampling correlation data for all years (worst months)...')

CORR_DFS = {}
for year in YEARS:
    dfs = []
    for mname, mnum in MONTHS.items():
        try:
            df = sample_all_vars(year, mname, mnum)
            df['Month'] = mname
            df['Year']  = year
            dfs.append(df)
        except Exception as e:
            print(f'  ⚠️  {year}/{mname}: {e}')
    if dfs:
        CORR_DFS[year] = pd.concat(dfs, ignore_index=True)
    print(f'  ✅ {year}')

print('✅ Correlation data sampled.')


In [ ]:
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm

VAR_COLS = ['VCI','TCI','VHI','NDVI','LST','Rainfall','Altitude']

# Combined correlation matrix across all years
all_data = pd.concat(CORR_DFS.values(), ignore_index=True)
corr_all = all_data[VAR_COLS].dropna().corr()

fig, ax = plt.subplots(figsize=(9, 7))
norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
sns.heatmap(
    corr_all,
    annot=True, fmt='.2f', ax=ax,
    cmap='RdYlGn', norm=norm,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 10}
)
ax.set_title('Correlation Matrix — Drought Indices & Environmental Variables\n'
             'Bilate River Watershed (2019–2025, May–Aug)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/correlation_matrix_all_years.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: correlation_matrix_all_years.png')


# Per-year correlation matrices
fig, axes = plt.subplots(2, 4, figsize=(28, 12))
fig.suptitle('Annual Correlation Matrices — Drought Indices & Environmental Variables\n'
             'Bilate River Watershed, 2019–2025',
             fontsize=14, fontweight='bold')

for ax, year in zip(axes.flatten(), YEARS):
    if year in CORR_DFS:
        corr = CORR_DFS[year][VAR_COLS].dropna().corr()
        sns.heatmap(corr, annot=True, fmt='.2f', ax=ax,
                    cmap='RdYlGn', norm=norm,
                    linewidths=0.5, linecolor='white',
                    annot_kws={'size': 8},
                    cbar=False)
        ax.set_title(str(year), fontweight='bold', fontsize=12)

# Hide empty subplots if YEARS < 8
for ax in axes.flatten()[len(YEARS):]:
    ax.set_visible(False)

plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/correlation_matrix_per_year.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: correlation_matrix_per_year.png')


## 📊 Step 10: Drought Spatial Distribution Composite Maps (per Year)

In [ ]:
# ─── Spatial distribution: multi-panel figure per year (4 months × 3 indices) ─
import requests
from PIL import Image as PILImage

def get_thumb(img, vis_params):
    """Fetch thumbnail array from GEE."""
    url = img.visualize(**vis_params).getThumbURL({
        'region': aoi_geom.bounds(),
        'dimensions': 200,
        'format': 'png'
    })
    resp = requests.get(url)
    return np.array(PILImage.open(io.BytesIO(resp.content)))


for year in YEARS:
    month_list = list(MONTHS.keys())
    indices_list = ['VCI','TCI','VHI']

    fig, axes = plt.subplots(3, 4, figsize=(18, 14))
    fig.suptitle(
        f'Spatial Distribution of Drought — {year}\nBilate River Watershed, Ethiopia',
        fontsize=15, fontweight='bold'
    )

    for row, idx_name in enumerate(indices_list):
        for col, mname in enumerate(month_list):
            ax = axes[row][col]
            try:
                img = ALL_INDICES[year][mname][idx_name]
                thumb = get_thumb(img, VIS[idx_name])
                ax.imshow(thumb)
            except Exception:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')

            if row == 0:
                ax.set_title(mname, fontsize=12, fontweight='bold')
            if col == 0:
                ax.set_ylabel(idx_name, fontsize=12, fontweight='bold', rotation=90)
                # Workaround: use text since axis is off
                ax.text(-0.08, 0.5, idx_name, transform=ax.transAxes,
                        fontsize=12, fontweight='bold',
                        va='center', ha='right', rotation=90)

    # Colorbar / legend
    legend_elements = [
        mpatches.Patch(facecolor='#d73027', label='Extreme (< 10)'),
        mpatches.Patch(facecolor='#fc8d59', label='Severe (10–20)'),
        mpatches.Patch(facecolor='#fee090', label='Moderate (20–30)'),
        mpatches.Patch(facecolor='#e0f3f8', label='Mild (30–40)'),
        mpatches.Patch(facecolor='#1a9850', label='No Drought (> 40)'),
    ]
    fig.legend(handles=legend_elements, loc='lower center',
               ncol=5, fontsize=10, title='Drought Class',
               title_fontsize=10, framealpha=0.9,
               bbox_to_anchor=(0.5, 0.0))

    # North arrow on first panel
    axes[0][0].annotate('N', xy=(0.92, 0.90), xycoords='axes fraction',
                        ha='center', fontsize=12, fontweight='bold')
    axes[0][0].annotate('', xy=(0.92, 0.88), xytext=(0.92, 0.80),
                        xycoords='axes fraction',
                        arrowprops=dict(arrowstyle='->', color='black', lw=2))

    fig.text(0.5, -0.01,
             'Data: MODIS MOD13Q1 & MOD11A2 | Source: NASA LP DAAC via Google Earth Engine',
             ha='center', fontsize=8, color='gray')

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    fname = f'{OUTPUT_DIR}/Spatial_Distribution_{year}.png'
    plt.savefig(fname, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved: {fname}')
    plt.close()


## 📥 Step 11: Download All Outputs as ZIP

In [ ]:
import zipfile
from google.colab import files

zip_path = 'BRW_Drought_Analysis_2019_2025.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(OUTPUT_DIR):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            zf.write(filepath, os.path.relpath(filepath, OUTPUT_DIR))

print(f'📦 ZIP created: {zip_path}')
print('📥 Downloading...')
files.download(zip_path)
print('✅ Download triggered!')


## 🖥️ Step 12: Check GEE Export Task Status

In [ ]:
# Monitor export tasks to Google Drive
print('📋 GEE Export Task Status:')
print(f'{'Name':<45} {'Status'}')
print('-' * 60)
for name, task in export_tasks:
    try:
        status = task.status().get('state', 'UNKNOWN')
    except:
        status = 'CHECK MANUALLY'
    print(f'{name:<45} {status}')

print('\n💡 Tip: GeoTIFF maps will appear in Google Drive → BRW_DroughtMaps folder')
print('      Open Google Drive to download individual GeoTIFF files.')


---
## 📋 Summary

| Output File | Description |
|---|---|
| `VCI/TCI/VHI_YEAR_MONTH.png` | Professional index maps (N arrow, legend, scale) |
| `Spatial_Distribution_YEAR.png` | 3×4 panel (3 indices × 4 months) per year |
| `VHI_area_coverage_by_month.png` | Stacked bar charts — drought area % per month |
| `VHI_drought_class_trends.png` | Extreme / Severe / Moderate trend lines |
| `VHI_annual_drought_summary.png` | Annual summary bar chart |
| `morans_I_heatmap.png` | Moran's I spatial autocorrelation heatmap |
| `morans_I_zscore_YEAR.png` | Z-score bar chart for peak drought year |
| `morans_i_summary.csv` | Full Moran's I statistics table |
| `correlation_matrix_all_years.png` | Combined correlation heatmap |
| `correlation_matrix_per_year.png` | Per-year correlation matrices |
| Google Drive → `BRW_DroughtMaps/` | Downloadable GeoTIFF rasters (VCI, TCI, VHI, classified) |

---
**References:**  
- Kogan (1995a, 1995b) — VCI, TCI, VHI indices  
- Bhuiyan et al. (2017) — VHI drought classification  
- Burka et al. (2024) — Spatial drought analysis methodology  
- Google Earth Engine: Gorelick et al. (2017)